<a href="https://colab.research.google.com/github/angadghatode/ISYS2001-Assignment2/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 - Personal Finance App

## Step 1: Understand the Problem
**The Problem:** Casual workers get paid by the hour, but when they buy something, the price tag doesn't show what it actually cost them. $120 sounds like a small number, but it can be a full shift of work after tax. With shifts that change every week, it's even harder to know what you can afford without setting your savings back.

**The User:** A casual retail or hospitality worker (like a uni student) who wants to save money but has unpredictable hours.

**The Solution:** A "What Does It Really Cost?" assistant. It learns your real take-home hourly pay from your past payslips. Then you type in something you want to buy, and it tells you the cost in hours of work and how many days it pushes back your savings goal. A friendly AI coach (Gemini) then asks "still want it?" and talks it through using your real numbers.

## Step 2: Inputs and Outputs
**Inputs:**
*   Payslip data: a .csv file with the columns Pay Date, Hours Worked, Gross Pay, Tax Withheld.

*   Savings goal: a goal amount and how much the user saves per week.

*   Purchase: the item name and price, typed in by the user.
*   Chat messages: questions like "can I afford this?" or "why is my tax higher this week?"

**Outputs:**
*   Cost-in-Hours Calculator (custom tool): a Python function you write yourself that works out:

    *  Average take-home hourly pay: (Gross − Tax) ÷ Hours
    * Hours of work the purchase costs: Price ÷ take-home hourly pay
    * Days the purchase delays the savings goal: Price ÷ daily savings rate

* AI Coach (Gemini): a friendly, honest friend that uses those numbers to reply, like "That's 6 hours of work and sets your goal back 9 days. Still want it?"
* Gradio interface: one page with the CSV upload, the purchase box, the results, and the chat.

## Step 3: Worked Example
Before coding, let's calculate the real cost of a purchase manually.

**1. The Fake Payslips**
*   Week 1: 15 hours, `$450` Gross, `$50` Tax (Net: `$400`)
*   Week 2: 20 hours, `$600` Gross, `$80` Tax (Net: `$520`)

**2. The User Inputs**
*   Target Purchase: Headphones (`$350`)
*   Savings Goal: `$100` per week

**3. The Manual Math**
*   Total Hours Worked: 15 + 20 = 35 hours
*   Total Net Pay: 400 + 520 = `$920`
*   Average Net Hourly Pay: `$920` / 35 hours = `$26.28` per hour
*   Cost in Hours: `$350` / `$26.28` = 13.3 hours of actual work
*   Daily Savings Rate: `$100` / 7 days = `$14.28` per day
*   Days Delayed: `$350` / `$14.28` = 24.5 days

## Step 4 - Pseudocode

```
FUNCTION calculate_real_cost(csv_data, item_price, weekly_savings):
    SET total_net_pay = 0
    SET total_hours = 0
    
    FOR EACH row IN csv_data:
        SET net_pay = row["Gross Pay"] - row["Tax Withheld"]
        ADD net_pay TO total_net_pay
        ADD row["Hours Worked"] TO total_hours
        
    IF total_hours EQUALS 0:
        RETURN "Error: No hours logged."
        
    SET average_takehome_hourly = total_net_pay / total_hours
    SET cost_in_hours = item_price / average_takehome_hourly
    
    SET daily_savings = weekly_savings / 7
    SET days_delayed = item_price / daily_savings
    
    RETURN average_takehome_hourly, cost_in_hours, days_delayed
```



In [8]:
%%writefile payslips.csv
Pay Date,Hours Worked,Gross Pay,Tax Withheld
2026-09-15,15,450,50
2026-09-22,20,600,80

Overwriting payslips.csv


## Step 5 (python)

In [ ]:
import pandas as pd

def calculate_real_cost(csv_path, item_price, weekly_savings):
    # Load the data
    df = pd.read_csv(csv_path)

    # Calculate net pay for each row and sum it up
    df['Net Pay'] = df['Gross Pay'] - df['Tax Withheld']
    total_net_pay = df['Net Pay'].sum()
    total_hours = df['Hours Worked'].sum()

    # Handle the edge case where no hours are logged
    if total_hours == 0:
        return "Error: No hours logged."

    # Do the math
    average_takehome_hourly = total_net_pay / total_hours
    cost_in_hours = item_price / average_takehome_hourly

    daily_savings = weekly_savings / 7
    days_delayed = item_price / daily_savings

    # Return a dictionary with the rounded results
    return {
        "Hourly Rate": round(average_takehome_hourly, 2),
        "Cost in Hours": round(cost_in_hours, 1),
        "Days Delayed": round(days_delayed, 1)
    }

# Test it right now to make sure it works!
result = calculate_real_cost('payslips.csv', 350, 100)
print(f"Calculation Result: {result}")

Calculation Result: {'Hourly Rate': np.float64(26.29), 'Cost in Hours': np.float64(13.3), 'Days Delayed': 24.5}


In [ ]:
%%writefile zero_hours_payslip.csv
Pay Date,Hours Worked,Gross Pay,Tax Withheld
2026-10-05,0,0,0

Overwriting zero_hours_payslip.csv


In [ ]:
## Step 6: Tests

# 1. Happy Path Test: Test the function against the normal dummy data we created yesterday.
# We know the math should equal 26.29 (or 26.28 depending on rounding), 13.3, and 24.5.
normal_result = calculate_real_cost('payslips.csv', 350, 100)
assert normal_result["Hourly Rate"] == 26.28 or normal_result["Hourly Rate"] == 26.29, f"Failed: Expected 26.28 or 26.29, got {normal_result['Hourly Rate']}"
assert normal_result["Cost in Hours"] == 13.3, f"Failed: Expected 13.3, got {normal_result['Cost in Hours']}"
assert normal_result["Days Delayed"] == 24.5, f"Failed: Expected 24.5, got {normal_result['Days Delayed']}"
print("Happy Path Test passed!")

# 2. Edge Case Test: Test what happens if the user uploads a CSV with zero hours worked.
error_result = calculate_real_cost('zero_hours_payslip.csv', 350, 100)
assert error_result == "Error: No hours logged.", f"Failed: Expected 'Error: No hours logged.', got {error_result}"
print("Edge Case Test passed!")

Happy Path Test passed!
Edge Case Test passed!


In [7]:
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata

# Safely load the API key
GOOGLE_API_KEY = userdata.get('GEMINIKEY')
genai.configure(api_key=GOOGLE_API_KEY)

def ask_finance_coach(purchase_item, calculation_results, user_question):
    # Set the Persona (Requirement 1)
    system_instruction = """You are a direct, pragmatic financial coach for a casual retail worker.
    Your goal is to help them understand the real cost of their purchases based on their actual take-home pay.
    Be conversational, empathetic about unpredictable shifts, but firm about the math.
    Do not give generic financial advice; strictly reference the calculated hours and days delayed provided in the prompt."""

    model = genai.GenerativeModel(
        model_name='gemini-flash-latest',
        system_instruction=system_instruction
    )

    # Ground the model in the user's real CSV data (Requirement 2)
    prompt = f"""
    The user wants to buy: {purchase_item}.
    According to their parsed payslips, their real take-home pay is ${calculation_results['Hourly Rate']} per hour.
    This purchase will cost them {calculation_results['Cost in Hours']} hours of standing on the retail floor.
    It will set their savings goal back by {calculation_results['Days Delayed']} days.

    The user asks: "{user_question}"
    """

    response = model.generate_content(prompt)
    return response.text

# Test the integration using the results from Step 5
ai_response = ask_finance_coach(
    purchase_item="Sony Headphones",
    calculation_results=normal_result,
    user_question="Is this purchase worth it, or should I wait until I get more shifts?"
)

print(ai_response)

Look, I get it. A solid pair of Sony headphones sounds amazing, especially when you need to zone out after a brutal shift. But let’s look at what this actually costs you in reality, not just the sticker price.

At your take-home rate of **$26.29 an hour**, those headphones equal **13.3 hours** of standing on the retail floor. That is nearly two full shifts of dealing with customers, folding stock, and being on your feet just to break even on this one purchase. 

On top of that, pulling the trigger right now sets your savings goal back by **24.5 days**. That’s almost an entire month of progress gone.

Given that casual retail hours can drop without warning, you shouldn't spend money banking on shifts you don't physically have on your roster yet. 

**My advice: Wait.** 

Don’t trade 13.3 hours of physical grind and lose 24.5 days of financial breathing room today on the *hope* that more shifts will show up later. Wait until those extra shifts are actually worked, the money hits your acco